In [63]:
import numpy as np
import torch

In [75]:
def get_W(beta, J=1, h=0):
    """
    Calculate the matrix W based on beta, J, and h.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).

    Returns:
        torch.Tensor: Matrix W.
    """
    sq_cosh = np.sqrt(np.cosh(beta * J))
    sq_sinh = np.sqrt(np.sinh(beta * J))
    W = torch.tensor([
        [sq_cosh, sq_sinh],
        [sq_cosh, -sq_sinh]
    ])
    return W


#可能Wh是正確的包含h的作法
def get_Wh(beta, J, h):
    """
    Calculate the matrix Wh incorporating the external field h.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength.
        h (float): External field strength.

    Returns:
        torch.Tensor: Matrix Wh.
    """
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)

    Wh = torch.tensor([
        [np.sqrt(cosh) * np.exp(beta * h / 2), np.sqrt(sinh) * np.exp(beta * h / 2)],
        [np.sqrt(cosh) * np.exp(-beta * h / 2), -np.sqrt(sinh) * np.exp(-beta * h / 2)]
    ])
    return Wh


# 上左右下
def get_T_bare(beta, J=1, h=0, get_W=get_W):
    """
    Calculate the bare transfer matrix T_bare.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).
        get_W (callable): Function to calculate W or Wh (default is get_W).

    Returns:
        torch.Tensor: Bare transfer matrix T_bare.
    """
    W = get_W(beta, J=J, h=h)
    T_bare = torch.einsum('ai,aj,ak,al->ijkl', W, W, W, W)
    return T_bare

def merge_y(Tup, Tdn, combine=False, trace=False):
    if trace:
        Tmerge = torch.einsum('abcd,defa->becf', Tup, Tdn)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0] * shape[1], shape[2] * shape[3])
    else:
        Tmerge = torch.einsum('abcd,defg->abecfg', Tup, Tdn)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0], shape[1] * shape[2], shape[3] * shape[4], shape[5])
            
    return Tmerge

def merge_y_truncate(Tup, Tdn, dcut, niter = 10):
    if ((Tup.shape[1] * Tdn.shape[1]) < dcut):
        return merge_y(Tup, Tdn, True)
    Tmerge_pure = torch.einsum('abcd,defg,achi,iejg->bfhj', Tup, Tdn, Tup, Tdn)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.contiguous().view(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1],Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    U, S, V = torch.svd_lowrank(Tmerge_pure, q=dcut, niter=niter)
    U = U.view(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    Tmerge = torch.einsum('abcd,defg,bfh,cei->ahig', Tup, Tdn, U, U)
    return Tmerge

def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        Tmerge = torch.einsum('abca,ecfe->bf', TL, TR)
    else:
        Tmerge = torch.einsum('abcd,ecfg->eabfgd', TL, TR)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge

def merge_x_truncate(TL, TR, dcut, niter = 10):
    if ((TL.shape[0] * TR.shape[0]) < dcut):
        return merge_x(TL, TR, True)
    Tmerge_pure = torch.einsum('abcd,ecfg,hbid,jifg->aehj', TL, TR, TL, TR)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.contiguous().view(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1],Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    U, S, V = torch.svd_lowrank(Tmerge_pure, q=dcut, niter=niter)
    U = U.view(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    Tmerge = torch.einsum('abcd,ecfg,aeh,dgi->hbfi', TL, TR, U, U)
    return Tmerge

In [65]:
MaxL = 4
Temp = np.linspace(2.26,4,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [66]:
%%time

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TL_Trace = merge_y(TL,TL,True,True)
        eigvals, eigvecs = torch.linalg.eigh(TL_Trace)
        W[j,i,:] = eigvals[-2:]
        del TL_Trace
        if j != MaxL:
            TL = merge_y(TL,TL,True)
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 2.33 s, sys: 1.79 s, total: 4.12 s
Wall time: 682 ms


<timed exec>:12: RuntimeWarning: divide by zero encountered in divide


In [67]:
Corr_len

array([[        inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,     

In [84]:
%%time
dcut = 3

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    print(TL)
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        print(TLx)
        TL_Trace = merge_y(TLx,TLx,True,True)
        eigvals, eigvecs = torch.linalg.eigh(TL_Trace)
        W[j,i,:] = eigvals[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        print(TL)
        TL = TL/torch.mean(torch.abs(TL))
        print(TL)
    break
E = -np.log(W)
Corr_len_2 = 1/(E[:,:,-2]-E[:,:,-1])

tensor([[[[2.4178, 0.0000],
          [0.0000, 1.0051]],

         [[0.0000, 1.0051],
          [1.0051, 0.0000]]],


        [[[0.0000, 1.0051],
          [1.0051, 0.0000]],

         [[1.0051, 0.0000],
          [0.0000, 0.4178]]]], dtype=torch.float64)
tensor([[[[ 6.0363e+00, -1.2209e-21,  1.9667e-08],
          [-3.0928e-21, -2.6055e+00, -8.7233e-01]],

         [[-3.1495e-21, -2.6055e+00,  8.7233e-01],
          [ 1.4374e+00, -3.1939e-22, -5.4466e-08]]],


        [[[-1.2209e-21,  3.4402e+00,  8.2086e-29],
          [-2.6055e+00,  3.3537e-21,  3.0560e-08]],

         [[-2.6055e+00,  3.3537e-21,  3.0560e-08],
          [-3.1939e-22,  1.4301e+00, -3.3307e-16]]],


        [[[ 1.9667e-08, -3.3307e-16,  1.4199e+00],
          [-8.7233e-01,  3.0560e-08, -1.0196e-07]],

         [[ 8.7233e-01,  3.0560e-08,  1.0196e-07],
          [-5.4466e-08, -5.5511e-17, -5.9025e-01]]]], dtype=torch.float64)
tensor([[[[ 3.7329e+01,  7.3555e-08,  6.1941e-08],
          [-3.7686e-08,  1.8378e+01, -2.673

<timed exec>:19: RuntimeWarning: divide by zero encountered in divide


In [69]:
Corr_len_2

array([[       inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,     